#### access to .env file

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

##### setting the client

In [2]:
import os
api_key = os.getenv("ANTHROPIC_API_KEY")

In [3]:
from anthropic import Anthropic
claude_client = Anthropic()

##### dataset

In [4]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

In [5]:
courses_raw

[{'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255}]

In [6]:
documents = []
url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
    course_url = f'{url_prefix}{course['path']}'

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1208

In [7]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

##### search engine

In [8]:
from minsearch import Index

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)

index.fit(documents)

In [9]:
question = 'I just discovered the course. Can I join now?'

search_results = index.search(
    question,
    boost_dict={'question': 2.0, 'section': 0.5},
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [10]:
[doc['question'] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'When will the course be offered next?',
 'I missed the first homework - can I still get a certificate?']

In [11]:
def search(question, course='llm-zoomcamp'):
    boost_dict = {'question': 2.0, 'section': 0.5}
    filter_dict = {'course': course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [12]:
search_results = search(question)

#### building prompt

In [13]:
# sytem prompt 

INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

In [14]:
# user prompt 

USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}
'''

In [15]:
# then the context has to be built: it is a formatted string with the retrieved documents from the search function.

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('')

    return '\n'.join(lines).strip()

In [16]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [17]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

#### adding the llm for the augmentation

In [18]:
response = claude_client.messages.create(
    model="claude-haiku-4-5-20251001", # È il più leggero/economico della famiglia attuale, perfetto per sperimentare.
    max_tokens=1024,
    messages=[{"role": "user", "content": prompt}]
)

In [19]:
response

Message(id='msg_01MCbQDeESeRVQvEszyYo4t4', container=None, content=[TextBlock(citations=None, text="# Answer\n\nYes, you can join the course now! \n\nHowever, here's what you should know about certificates:\n\n- **If you want a certificate**, you need to submit your capstone project while submissions are still being accepted\n- **Self-paced completion doesn't qualify for a certificate** — you can only earn one by completing the course with a live cohort, since certificates require peer-reviewing 3 capstone projects (which only happens during active course runs)\n\n**Next offering:** Summer 2025\n\nYou can start learning and submitting homework immediately without needing to wait for a confirmation email. Registration is optional and just helps gauge interest.", type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5

In [20]:
print(response.content[0].text)

# Answer

Yes, you can join the course now! 

However, here's what you should know about certificates:

- **If you want a certificate**, you need to submit your capstone project while submissions are still being accepted
- **Self-paced completion doesn't qualify for a certificate** — you can only earn one by completing the course with a live cohort, since certificates require peer-reviewing 3 capstone projects (which only happens during active course runs)

**Next offering:** Summer 2025

You can start learning and submitting homework immediately without needing to wait for a confirmation email. Registration is optional and just helps gauge interest.


###### pricing of anthropic

Input token = tutto quello che mandi al modello: il tuo prompt, il contesto, il system prompt.
Output token = tutto quello che il modello genera in risposta.

In [21]:
input_price = 1.00 / 1_000_000
output_price = 5.00 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.001078

In [22]:
# la chiamata api in termini di system prompt e user prompt!

message = [
        {"role": "user", "content": prompt}
    ]

response = claude_client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    system=INSTRUCTIONS,        # ← separato
    messages=message
)

In [23]:
print(response.content[0].text)

Yes, you can join now! 

However, there's an important detail to know: **if you want to receive a certificate, you need to submit your project while submissions are still being accepted.**

A few additional points that might be helpful:

- You don't need to wait for a confirmation email to start learning and submitting assignments
- Registration is optional and just helps gauge interest
- Certificates are only awarded for completing the course with a "live" cohort (not for self-paced learning)
- To get a certificate, you'll need to complete and submit the capstone project, then participate in peer-reviewing 3 other capstone projects

Feel free to start learning right away!


==> in una situazione più realistica-complessa, tipica degli chatbot, bisognerebbe tenere conto della history
user_message = {"role": "user", "content": prompt}

history = []

==> ogni turno aggiungi il messaggio utente
history.append(user_message)

==> chiami il modello con tutta la history
response = claude_client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    system=INSTRUCTIONS,
    messages=history
)


assistant_message = {"role": "assistant", "content": response.content[0].text}

==> poi aggiungi anche la risposta del modello alla history
history.append(assistant_message)

###### llm function

In [24]:
def llm (instructions, user_prompt, model="claude-haiku-4-5-20251001"):
    message = [
        {"role": "user", "content": user_prompt}
    ]

    response = claude_client.messages.create(
        model=model,
        max_tokens=1024,
        system=instructions,  # INSTRUCTIONS
        messages=message
    )

    return response.content[0].text

##### wiring all the 3 components

In [25]:
def rag(query, model="claude-haiku-4-5-20251001"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [26]:
answer = rag(question)
print(answer)

Yes, you can join the course now! 

However, there's an important detail regarding certificates: **if you want to receive a certificate, you need to submit your project while submissions are still being accepted.**

Also note that:
- You don't need to wait for a confirmation email to start - you're accepted and can begin learning and submitting homework right away
- Certificates are only awarded to those who complete the course with a "live" cohort (not self-paced mode)
- To get a certificate, you'll need to pass the Capstone project and participate in peer-reviewing 3 capstone projects from other students

So go ahead and start whenever you're ready, just keep in mind the submission deadlines if you're interested in getting certified!


==> now we wanna clean the code: we create ingest.py and rag_helper.py